Chargement des bibliothèques

In [ ]:
# --- Import des bibliothèques fondamentales ---
# pandas : pour la manipulation de données (DataFrames)
# numpy : pour les opérations numériques (souvent utilisé en arrière-plan par pandas)
# matplotlib & seaborn : pour la visualisation de données
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration de l'environnement du notebook ---
# %matplotlib inline : commande magique pour afficher les graphiques directement dans le notebook
# pd.set_option : pour configurer le comportement de pandas, ici pour afficher jusqu'à 50 colonnes
# plt.style.use : pour définir un style visuel par défaut pour tous les graphiques (ggplot est un style populaire)
%matplotlib inline
pd.set_option('display.max_columns', 50)
plt.style.use('ggplot')

Chargement du dataset

In [ ]:
import pandas as pd
from io import StringIO
import csv

# --- Chargement robuste des données CSV ---
# Le fichier de log peut contenir des erreurs de formatage (lignes avec un nombre incorrect de colonnes).
# Cette cellule met en place une méthode de lecture robuste pour gérer ces erreurs sans planter.

# Chemin vers le fichier de données
log_file = "logs/user_events_with_geoip_25k.csv"

print(f"Tentative de lecture du fichier '{log_file}'...")

# Liste pour stocker uniquement les lignes valides du fichier
cleaned_lines = []
num_bad_lines = 0

try:
    with open(log_file, 'r', encoding='utf-8') as f:
        # 1. Lire l'en-tête pour déterminer le nombre de colonnes attendu
        header_line = f.readline()
        cleaned_lines.append(header_line)
        
        # On compte le nombre de séparateurs (virgules) pour déduire le nombre de champs
        expected_fields = header_line.count(',') + 1
        print(f"Nombre de champs attendus par ligne : {expected_fields}")

        # 2. Parcourir le reste du fichier ligne par ligne
        for i, line in enumerate(f, start=2):
            # On vérifie si la ligne a le bon nombre de champs
            current_fields = line.count(',') + 1
            
            if current_fields == expected_fields:
                # Si la ligne est correcte, on la conserve
                cleaned_lines.append(line)
            else:
                # Sinon, on la signale et on l'ignore
                num_bad_lines += 1
                continue # On passe à la ligne suivante

    print(f"\nLecture terminée. {num_bad_lines} lignes mal formées ont été ignorées.")

    # 3. Créer un "fichier en mémoire" à partir des lignes nettoyées
    # StringIO permet à pandas de lire une chaîne de caractères comme si c'était un fichier sur le disque
    cleaned_csv_data = "".join(cleaned_lines)
    csv_file_in_memory = StringIO(cleaned_csv_data)
    
    # 4. Charger les données en mémoire dans un DataFrame pandas
    df = pd.read_csv(csv_file_in_memory, sep=',')
    
    print(f"Fichier '{log_file}' chargé avec succès dans Pandas.")
    print(f"Dimensions du DataFrame : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    
except FileNotFoundError:
    print(f"ERREUR: Le fichier '{log_file}' n'a pas été trouvé. Veuillez vérifier le chemin.")
    df = None
except Exception as e:
    print(f"Une autre erreur est survenue : {e}")
    df = None

# --- Inspection initiale du DataFrame ---
if df is not None:
    # Afficher les premières lignes pour avoir un aperçu des données
    print("\nAperçu des données brutes :")
    display(df.head())

    # .info() donne un résumé concis : nom des colonnes, types de données, et nombre de valeurs non nulles
    print("\nInformations sur le DataFrame :")
    df.info()

In [ ]:
# --- Nettoyage et standardisation des noms de colonnes ---
# Il est recommandé de travailler avec une copie du DataFrame original pour garder une sauvegarde.
df_clean = df.copy()

# 1. Normalisation : Mettre tous les noms de colonnes en minuscules
# Cela évite les erreurs dues à la casse (ex: 'Timestamp' vs 'timestamp')
df_clean.columns = df_clean.columns.str.lower()

# 2. Renommage : Rendre les noms de colonnes plus clairs et plus courts
# On utilise un dictionnaire pour mapper les anciens noms aux nouveaux.
column_mapping = {
    'hasheduserid': 'user_id',
    'hashedip': 'ip_address',
    'eventtype': 'event_type',
    'authtype': 'auth_type',
    'hasheduseragent': 'user_agent_hash', # On précise que c'est un hash
    'clientid': 'client_id'
}
df_clean.rename(columns=column_mapping, inplace=True)

print("Colonnes après nettoyage et renommage :")
print(df_clean.columns.tolist())

In [ ]:
# --- Traitement de la colonne 'timestamp' ---
# La colonne de temps est l'une des plus importantes. Il faut la convertir
# d'une chaîne de caractères à un objet datetime pour pouvoir effectuer des opérations temporelles.

# `pd.to_datetime` est la fonction dédiée pour cette conversion.
# `errors='coerce'` est une option de sécurité : si une date ne peut pas être convertie,
# elle sera remplacée par `NaT` (Not a Time), au lieu de provoquer une erreur.
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce')

# --- Vérification et tri ---
# On vérifie que la conversion s'est bien passée et combien de valeurs sont devenues invalides.
print(f"Type de la colonne 'timestamp' après conversion : {df_clean['timestamp'].dtype}")
print(f"Nombre de timestamps invalides (NaT) : {df_clean['timestamp'].isnull().sum()}")

# Trier le DataFrame par timestamp est essentiel pour analyser la séquence des événements
# et pour les calculs temporels (ex: temps entre deux connexions).
df_clean.sort_values(by='timestamp', inplace=True)
# Après le tri, l'index du DataFrame est désordonné. On le réinitialise.
df_clean.reset_index(drop=True, inplace=True)

print("\nAperçu des données avec timestamp converti et trié :")
display(df_clean.head())

In [ ]:
# --- Gestion des valeurs manquantes (NaN) ---

# 1. Identifier les valeurs manquantes
# .isnull().sum() est une méthode efficace pour compter le nombre de valeurs manquantes dans chaque colonne.
print("Valeurs manquantes par colonne avant traitement :")
print(df_clean.isnull().sum())

# 2. Stratégie de remplissage (Imputation)
# Pour les colonnes catégorielles, remplacer les NaN par une nouvelle catégorie comme 'inconnu' est une bonne pratique.
# Cela conserve l'information que la donnée était manquante.
fill_values = {
    'auth_type': 'inconnu',
    'city': 'inconnu',
    'country': 'inconnu'
}
df_clean.fillna(fill_values, inplace=True)

print("\nValeurs manquantes après remplissage des données catégorielles :")
print(df_clean.isnull().sum())

# 3. Suppression des lignes critiques
# Si des identifiants essentiels comme `user_id` ou `ip_address` sont manquants,
# la ligne est inutilisable pour notre analyse de graphe. Il est donc préférable de la supprimer.
df_clean.dropna(subset=['user_id', 'ip_address'], inplace=True)
print(f"\nNouvelles dimensions après suppression des lignes avec IDs manquants : {df_clean.shape}")

In [ ]:
# --- Analyse exploratoire de base (EDA) ---
# L'objectif est de comprendre la distribution des données avant de continuer.

print("Répartition des types d'événements (Top 15) :")
# .value_counts() est une méthode très utile pour compter les occurrences de chaque catégorie.
event_counts = df_clean['event_type'].value_counts()
print(event_counts.head(15))

# --- Visualisation ---
# Un graphique est souvent plus parlant qu'un tableau de chiffres.
plt.figure(figsize=(12, 8))
sns.barplot(y=event_counts.head(15).index, x=event_counts.head(15).values, orient='h')
plt.title('Top 15 des types d\'événements')
plt.xlabel('Nombre d\'occurrences')
plt.ylabel('Type d\'événement')
plt.show()

print("\nRépartition des types d'authentification :")
print(df_clean['auth_type'].value_counts())

In [ ]:
# On se concentre sur les événements liés à la connexion
login_events = [
    'login',
    'login_matching_password',
    'login_not_matching_password',
    'login_unknown_identifier'
]

# Filtrer le DataFrame pour ne garder que ces événements
df_logins = df_clean[df_clean['event_type'].isin(login_events)].copy()

# Créer une colonne 'login_status'
def get_login_status(event):
    if event in ['login', 'login_matching_password']:
        return 'success'
    elif event in ['login_not_matching_password', 'login_unknown_identifier']:
        return 'failure'
    else:
        return 'other'

df_logins['login_status'] = df_logins['event_type'].apply(get_login_status)

print("Répartition des statuts de connexion :")
print(df_logins['login_status'].value_counts())

print("\nAperçu du DataFrame des connexions :")
display(df_logins.head())

In [ ]:
# ===================================================================
# Cellule : Ingénierie de caractéristiques temporelles et comportementales
# ===================================================================

print("Début de l'ingénierie de caractéristiques avancées...")

# S'assurer que le timestamp est au bon format et trié
df_logins['timestamp'] = pd.to_datetime(df_logins['timestamp'])
df_logins.sort_values('timestamp', inplace=True)

# --- 1. Fréquence de connexion (par utilisateur et par IP) ---
# On calcule le temps moyen (en secondes) entre les connexions successives.
# La fréquence sera l'inverse de ce temps moyen.

# Pour les utilisateurs
time_diffs_user = df_logins.groupby('user_id')['timestamp'].diff().dt.total_seconds()
# Remplacer les NaT (première occurrence) par 0 pour ne pas fausser la moyenne
mean_time_user = time_diffs_user.groupby(df_logins['user_id']).mean()
# Fréquence = 1 / temps moyen. On ajoute 1 pour éviter la division par zéro et gérer les événements uniques.
user_connection_frequency = 1 / (mean_time_user + 1)
df_logins['user_conn_freq'] = df_logins['user_id'].map(user_connection_frequency).fillna(0)

# Pour les IPs
time_diffs_ip = df_logins.groupby('ip_address')['timestamp'].diff().dt.total_seconds()
mean_time_ip = time_diffs_ip.groupby(df_logins['ip_address']).mean()
ip_connection_frequency = 1 / (mean_time_ip + 1)
df_logins['ip_conn_freq'] = df_logins['ip_address'].map(ip_connection_frequency).fillna(0)

# --- 2. Diversité du User-Agent (par utilisateur et par IP) ---
# On compte le nombre de user agents uniques par utilisateur et par IP.

user_ua_diversity = df_logins.groupby('user_id')['user_agent_hash'].nunique()
df_logins['user_ua_diversity'] = df_logins['user_id'].map(user_ua_diversity)

ip_ua_diversity = df_logins.groupby('ip_address')['user_agent_hash'].nunique()
df_logins['ip_ua_diversity'] = df_logins['ip_address'].map(ip_ua_diversity)


print("Nouvelles caractéristiques ajoutées :")
print(df_logins[['user_id', 'ip_address', 'user_conn_freq', 'ip_conn_freq', 'user_ua_diversity', 'ip_ua_diversity']].head())

print("\nDescription des nouvelles caractéristiques :")
display(df_logins[['user_conn_freq', 'ip_conn_freq', 'user_ua_diversity', 'ip_ua_diversity']].describe())

In [ ]:
# Nom du fichier de sortie
output_file = "logs_events_clean.csv"

# Sauvegarder le DataFrame nettoyé et enrichi
df_logins.to_csv(output_file, index=False)

print(f"Le fichier de données prétraitées a été sauvegardé sous : '{output_file}'")